# HW1b - Photoshop from scratch

**Convolutions and colour spaces.** Chapter 6 (CNNs), attached to **L16 CNN Foundations**.
No PyTorch, no GPU, no training.

> **Notebook 1 of 2.** This one builds the tools. **HW1c** takes them to a real liver
> biopsy and watches them fall apart, which is the argument for the rest of the chapter.
>
> This is the **task** notebook. Solutions: `HW1b_photoshop_solution.ipynb`.

By the end you will have built:

1. a working subset of Photoshop's **Filter** menu, one filter at a time,
2. the **blurred-background thumbnail** every lo-fi music upload on YouTube uses,
3. the **Hue/Saturation** adjustment panel.

### Rules

- **No `scipy.signal.convolve2d`, no `cv2`.** You are writing the convolution.
- `numpy`, `matplotlib` and `PIL` only.
- Every number in the text is **computed in a cell**, never typed by hand.

### Contents

| Part | | |
|---|---|---|
| 0 | 🧀 | the engine - and what the kernel is actually doing, step by step |
| 1 | 🧀 | the kernel zoo, one filter at a time, with its knobs |
| 2 | 🧀🧀 | the thumbnail generator, built up one layer at a time |
| 3 | 🧀🧀 | HSV - separating "which colour" from "how bright" |

In [ ]:
import time
from pathlib import Path
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv
from matplotlib.patches import Rectangle
from PIL import Image

SEED = 509
rng = np.random.default_rng(SEED)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"

CANVAS = (720, 1280)          # the thumbnail we are heading towards, in Part 2
RAW = ("https://raw.githubusercontent.com/HaykTarkhanyan/"
       "python_math_ml_course/main/ml/ch6_cnn/")


def load_image(relpath: str) -> np.ndarray:
    """Load a repo image as float RGB in [0, 1].

    Prefers the local checkout. On Colab (no repo) it downloads once and caches.
    If both routes fail it RAISES - never returns a placeholder.
    """
    local = Path(relpath)
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        url = RAW + relpath.replace("\\", "/")
        req = Request(url, headers={"User-Agent": "python-math-ml-course/hw1b"})
        with urlopen(req, timeout=60) as r:          # raises on 404 / no network
            local.write_bytes(r.read())
        print(f"downloaded {url} -> {local}")
    return np.asarray(Image.open(local).convert("RGB"), dtype=float) / 255.0


def show(images, titles, ncols=None, size=3.0, suptitle=None):
    """Plot a labelled row/grid of images."""
    n = len(images)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * 1.14 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(np.clip(im, 0, 1))
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.01)
    fig.tight_layout()
    plt.show()


def to_gray(rgb):
    """Rec. 601 luma - the weights match how the eye weights R, G, B."""
    return rgb @ np.array([0.299, 0.587, 0.114])


print("setup ok")

---
# Part 0 - the engine 🧀

L16 defines 2D convolution as: slide the kernel over the image, and at every stop take the
dot product of the kernel with the window sitting under it.

$$s_{ij}=\sum_{m,n} I(i{+}m{-}1,\;j{+}n{-}1)\,w(m,n)$$

Note there is **no kernel flip**. Strictly this is *cross-correlation*, and it is what every
CNN library actually computes - L16's footnote covers why the distinction stops mattering
once the weights are learned. Here they are not learned, so match the slides exactly.

### Task 0.1 - write it 🧀

The direct transcription of the formula: loop over output pixels, and for each one sum the
`k x k` window times the kernel.

Support the three cases from L16 Section 4:

- `padding="valid"` - no padding, the map shrinks
- `padding="same"` - pad by `k // 2` so the map keeps its size
- `stride=s` - move the kernel `s` pixels at a time

Output size is $o = \lfloor (i - k + 2p)/s \rfloor + 1$.

In [ ]:
def _pad(img, pad, mode):
    if pad == 0:
        return img
    return np.pad(img, ((pad, pad), (pad, pad)),
                  mode="edge" if mode == "edge" else "constant")


def convolve2d_naive(img, kernel, padding="same", stride=1, pad_mode="edge"):
    """Loop over OUTPUT PIXELS. The definition, written out. Correct and slow."""
    # YOUR CODE HERE - loop over output pixels; honour padding, stride and pad_mode
    raise NotImplementedError("loop over output pixels; honour padding, stride and pad_mode")

print("convolve2d_naive defined")

### Task 0.2 - watch it slide 🧀

Before running this on a photo, watch what the kernel does on something small enough to read.

Below is a **6x6 image** with a vertical edge in it: dark on the left, bright on the right.
The kernel is **Sobel X**, the vertical-edge detector.

At every stop you see four things: the window under the kernel, the kernel itself, the two
multiplied cell by cell, and the single number that lands in the output.

In [ ]:
def grid_plot(ax, arr, title, highlight=None, cmap="gray", vmin=None, vmax=None,
              fmt="{:.0f}", fontsize=9, textcolor=None):
    """Draw a small matrix as a heatmap with the numbers written in each cell."""
    ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, vmin=vmin, vmax=vmax)
    h, w = arr.shape
    for y in range(h):
        for x in range(w):
            v = arr[y, x]
            if not np.isfinite(v):        # not computed yet - leave the cell blank
                continue
            if textcolor is None:
                lo = vmin if vmin is not None else arr.min()
                hi = vmax if vmax is not None else arr.max()
                rel = 0.5 if hi == lo else (v - lo) / (hi - lo)
                col = "white" if rel < 0.5 else "black"
            else:
                col = textcolor
            ax.text(x, y, fmt.format(v), ha="center", va="center",
                    fontsize=fontsize, color=col)
    if highlight is not None:
        y0, x0, kh, kw = highlight
        ax.add_patch(Rectangle((x0 - 0.5, y0 - 0.5), kw, kh, fill=False,
                               edgecolor="#D90012", lw=2.6))
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


# a 6x6 image with a vertical edge: dark left, bright right
IMG6 = np.full((6, 6), 20.0)
IMG6[:, 3:] = 200.0
IMG6[1, 1] = 60.0          # a little texture so the flat region is not perfectly flat
IMG6[4, 4] = 160.0

SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

print("input image (6x6):"); print(IMG6.astype(int))
print("\nSobel X kernel:"); print(SOBEL_X.astype(int))

In [ ]:
# walk the kernel across the top row of output positions, 'valid' padding
positions = [(0, 0), (0, 1), (0, 2), (0, 3)]
out_valid = convolve2d_naive(IMG6, SOBEL_X, padding="valid")

fig, axes = plt.subplots(len(positions), 4, figsize=(13, 2.9 * len(positions)))
for row, (oy, ox) in enumerate(positions):
    win = IMG6[oy:oy + 3, ox:ox + 3]
    prod = win * SOBEL_X
    running = np.full_like(out_valid, np.nan)
    running[:oy, :] = out_valid[:oy, :]
    running[oy, :ox + 1] = out_valid[oy, :ox + 1]

    grid_plot(axes[row, 0], IMG6, f"image, window at ({oy},{ox})",
              highlight=(oy, ox, 3, 3), vmin=0, vmax=255)
    grid_plot(axes[row, 1], SOBEL_X, "kernel (Sobel X)", cmap="coolwarm",
              vmin=-2, vmax=2, textcolor="black")
    grid_plot(axes[row, 2], prod, f"window x kernel  ->  sum = {prod.sum():.0f}",
              cmap="coolwarm", vmin=-450, vmax=450, textcolor="black")
    grid_plot(axes[row, 3], running, "output so far (blank = not yet computed)",
              cmap="coolwarm", vmin=-900, vmax=900, textcolor="black",
              highlight=(oy, ox, 1, 1))
fig.suptitle("One kernel, four stops. The red box is where the kernel is standing.",
             fontsize=12, y=1.002)
fig.tight_layout(); plt.show()

Read the third column. The kernel's left column is negative and its right column is positive,
so the sum is:

- **near zero** where the window sits entirely inside a flat region - the negatives and the
  positives cancel,
- **large** where the window straddles the edge - the bright side survives, the dark side
  contributes almost nothing.

That is the entire idea of an edge detector, and you can read it off six numbers. It is also
why the kernel's weights **summing to zero** matters: a zero-sum kernel is blind to flat
brightness and only reports change.

In [ ]:
print("full 'valid' output (4x4):")
print(convolve2d_naive(IMG6, SOBEL_X, padding="valid").astype(int))
print("\nColumn 2 is the edge (between input columns 2 and 3). The rest is near zero.")
print("\nWhat the same kernel gives on a completely flat image:")
print(convolve2d_naive(np.full((6, 6), 128.0), SOBEL_X, padding="valid").astype(int))

### Task 0.3 - padding, in 2D 🧀

A `3x3` kernel on a `6x6` image gives a `4x4` output - the map **shrinks**, because the kernel
cannot hang off the edge. Padding is how you keep the size.

Two choices, and they are not equivalent:

- **zeros** - honest but it invents a black border, so a blur darkens the edges,
- **edge replication** - copies the border pixel outwards, so a blur leaves them alone.

Look at what each does to the *corner values* below.

In [ ]:
pz = np.pad(IMG6, ((1, 1), (1, 1)), mode="constant")
pe = np.pad(IMG6, ((1, 1), (1, 1)), mode="edge")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
grid_plot(axes[0], IMG6, "original 6x6", vmin=0, vmax=255)
grid_plot(axes[1], pz, "pad 1 with ZEROS -> 8x8", vmin=0, vmax=255)
grid_plot(axes[2], pe, "pad 1 with EDGE -> 8x8", vmin=0, vmax=255)
fig.tight_layout(); plt.show()

box3 = np.ones((3, 3)) / 9
z = convolve2d_naive(IMG6, box3, padding="same", pad_mode="zero")
e = convolve2d_naive(IMG6, box3, padding="same", pad_mode="edge")
print("3x3 box blur, 'same' padding. Top-left 3x3 corner of the result:\n")
print("zero padding:\n", z[:3, :3].astype(int))
print("\nedge padding:\n", e[:3, :3].astype(int))
print(f"\nThe corner pixel is {z[0,0]:.0f} with zeros and {e[0,0]:.0f} with edge replication.")
print("The true local average there is about 20 - zero padding invented a dark rim.")

### Task 0.4 - stride, in 2D 🧀

Stride is how far the kernel jumps between stops. `stride=1` visits every pixel; `stride=2`
visits every other one, so the output is roughly half the size on each axis.

The formula $o = \lfloor (i - k + 2p)/s \rfloor + 1$ is just bookkeeping for that.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
for ax, st in zip(axes, (1, 2, 3)):
    o = convolve2d_naive(IMG6, box3, padding="valid", stride=st)
    grid_plot(ax, o, f"stride={st}  ->  output {o.shape[0]}x{o.shape[1]}",
              vmin=0, vmax=255, fontsize=8)
fig.suptitle("Same image, same kernel, three strides", fontsize=12, y=1.02)
fig.tight_layout(); plt.show()

print(f"{'k':>3}{'padding':>9}{'stride':>8}{'output':>10}{'formula':>10}")
for (k, pad, st) in [(3, "same", 1), (3, "valid", 1), (3, "valid", 2),
                     (5, "same", 1), (5, "same", 3)]:
    ker = np.ones((k, k)) / (k * k)
    got = convolve2d_naive(IMG6, ker, padding=pad, stride=st).shape
    p = k // 2 if pad == "same" else 0
    want = ((6 - k + 2 * p) // st + 1,) * 2
    assert got == want, f"k={k} {pad} stride={st}: got {got}, formula says {want}"
    print(f"{k:>3}{pad:>9}{st:>8}{str(got):>10}{str(want):>10}")
print("\nformula matches the implementation in all 5 cases  OK")

### Task 0.5 - prove the engine 🧀

Two checks, both `assert`s. If the engine is wrong the notebook stops here rather than
producing pretty but meaningless pictures for the next hour.

In [ ]:
# 1. the identity kernel must be a no-op
_img = rng.random((16, 16))
_identity = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float)
assert np.allclose(convolve2d_naive(_img, _identity), _img), "identity must be a no-op"

# 2. the worked example from the L16 slide - through YOUR function, not a special case
I = np.array([[1, 2, 0], [3, 1, 1], [0, 2, 4]], dtype=float)
W = np.array([[0, 1], [2, 2]], dtype=float)
S_expected = np.array([[10, 4], [5, 13]], dtype=float)
S = convolve2d_naive(I, W, padding="valid")
assert np.array_equal(S, S_expected), f"expected {S_expected.tolist()}, got {S.tolist()}"

print("identity kernel  OK")
print(f"L16 slide example OK  ->  S = {S.astype(int).tolist()}")

### Task 0.6 - the same thing, but fast 🧀

`convolve2d_naive` runs a Python loop once per output pixel. On a megapixel photo that is a
million iterations before any arithmetic happens.

Flip the loops: instead of looping over pixels and touching `k x k` weights, loop over the
**`k x k` weights** and let NumPy apply each one to the whole image at once. A 3x3 kernel
becomes 9 whole-image operations instead of a million small ones.

**This is not a better algorithm** - it performs exactly the same multiply-adds. It only
moves the loop from Python into NumPy's C. Hold on to that distinction; Part 2 shows a change
that *is* algorithmic, and one that is not.

In [ ]:
def convolve2d_fast(img, kernel, padding="same", stride=1, pad_mode="edge"):
    """Loop over KERNEL TAPS, vectorised over the image. Same answer, much faster."""
    # YOUR CODE HERE - loop over the k*k kernel taps instead, vectorised over the whole image
    raise NotImplementedError("loop over the k*k kernel taps instead, vectorised over the whole image")

_k = rng.random((5, 5)); _k /= _k.sum()
for pad in ("same", "valid"):
    for st in (1, 2):
        a = convolve2d_naive(_img, _k, padding=pad, stride=st)
        b = convolve2d_fast(_img, _k, padding=pad, stride=st)
        assert a.shape == b.shape and np.allclose(a, b, atol=1e-12), \
            f"engines disagree at padding={pad} stride={st}"
print("naive and fast agree to 1e-12 across padding and stride  OK")

img_t = rng.random((400, 400))
for k in (3, 9):
    ker = np.ones((k, k)) / (k * k)
    t0 = time.perf_counter(); convolve2d_naive(img_t, ker); tn = time.perf_counter() - t0
    t0 = time.perf_counter(); convolve2d_fast(img_t, ker); tf = time.perf_counter() - t0
    print(f"k={k}: naive {tn:6.3f}s   fast {tf:6.3f}s   ({tn/tf:5.0f}x)")

---
# Part 1 - the kernel zoo, one filter at a time 🧀

Every filter below is a real Photoshop menu item, and they are not *analogies* for it - they
are the same arithmetic.

| Photoshop menu | what it is |
|---|---|
| Filter > Blur > Box Blur | `ones((k,k)) / k**2` |
| Filter > Blur > Gaussian Blur | a sampled 2D Gaussian |
| Filter > Blur > Motion Blur | a 1-D line, at an angle |
| Filter > Sharpen > Sharpen | unsharp mask |
| Filter > Stylize > Emboss | asymmetric 3x3 |
| Filter > Stylize > Find Edges | Sobel magnitude |
| Filter > Other > Custom | **literally a 5x5 kernel entry grid**, plus scale and offset |

That last row is worth sitting with. Photoshop's *Custom* dialog asks you to type 25 numbers
into a grid, divide by a scale and add an offset. You are not modelling that feature. You are
reimplementing it.

We take these **one at a time**, and for each one we turn its knob to see what the knob does.

In [ ]:
pom_full = load_image("fig/src_pomegranate.jpg")
# a smaller working copy keeps the pure-numpy convolutions responsive
pom = np.asarray(Image.fromarray((pom_full * 255).astype(np.uint8)).resize((480, 360),
                 Image.LANCZOS), float) / 255
pom_g = to_gray(pom)
print(f"working image: {pom.shape} (from {pom_full.shape})")
show([pom, pom_g], ["colour", "grayscale (Rec. 601 luma)"], ncols=2, size=4.0)

## 1.1 Box Blur - the simplest possible kernel 🧀

Every weight identical, summing to 1. Each output pixel is the plain average of the `k x k`
square around it.

The knob is `k`. Watch two things as it grows: the image gets softer, and the **shape** of the
softening becomes visible - a box kernel smears square, which is why real blurs are not boxes.

In [ ]:
def box_kernel(k):
    # YOUR CODE HERE - k by k, every weight equal, summing to 1
    raise NotImplementedError("k by k, every weight equal, summing to 1")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, k in zip(axes, (3, 5, 9, 15)):
    grid_plot(ax, box_kernel(k), f"box kernel k={k}", cmap="viridis",
              fmt="{:.3f}" if k <= 5 else "", fontsize=7)
fig.suptitle("The kernels themselves: every weight the same, all summing to 1",
             fontsize=12, y=1.03)
fig.tight_layout(); plt.show()

ks = (3, 9, 21, 41)
outs = [convolve2d_fast(pom_g, box_kernel(k)) for k in ks]
show([pom_g] + outs, ["original"] + [f"box blur, k={k}" for k in ks], ncols=5, size=2.7,
     suptitle="Turning the box-blur knob")
for k, o in zip(ks, outs):
    print(f"k={k:>3}: {k*k:>5,} taps per pixel,  "
          f"local contrast left = {np.std(pom_g - o):.4f}")

## 1.2 Gaussian Blur - the blur that has no direction 🧀

A box blur weights a pixel 20 away exactly as much as its neighbour, and stops abruptly. A
**Gaussian** falls off smoothly and is *rotationally symmetric*: it has no preferred direction,
so it does not smear things into squares.

The knob is `sigma`, the width of the bell. The kernel size follows from it - you need about
`±3 sigma` of it before the weights are negligible, so `k = 6*sigma + 1`.

Look at the kernel heatmaps: a **round** blob, not a square one.

In [ ]:
def gaussian_kernel(sigma, truncate=3.0):
    """2D Gaussian, truncated at +/- truncate*sigma, normalised to sum 1."""
    # YOUR CODE HERE - sample a 2D Gaussian out to +/- truncate*sigma, then normalise to sum 1
    raise NotImplementedError("sample a 2D Gaussian out to +/- truncate*sigma, then normalise to sum 1")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, s in zip(axes, (1, 2, 3, 5)):
    k = gaussian_kernel(s)
    grid_plot(ax, k, f"sigma={s}  ->  {k.shape[0]}x{k.shape[0]}", cmap="viridis",
              fmt="", fontsize=6)
fig.suptitle("Gaussian kernels: round, smooth, and they grow with sigma", fontsize=12, y=1.03)
fig.tight_layout(); plt.show()

sigmas = (1, 2, 4, 8)
gouts = [convolve2d_fast(pom_g, gaussian_kernel(s)) for s in sigmas]
show([pom_g] + gouts, ["original"] + [f"gaussian, sigma={s}" for s in sigmas],
     ncols=5, size=2.7, suptitle="Turning the Gaussian knob")

### Box vs Gaussian at matched width

Put them side by side at the same kernel size. The box blur leaves **square** artefacts and a
harsher transition; the Gaussian is smooth. Same cost, better result - which is why the
Gaussian is the default everywhere.

In [ ]:
K = 25
b = convolve2d_fast(pom_g, box_kernel(K))
g = convolve2d_fast(pom_g, gaussian_kernel(K / 6.0))
show([pom_g, b, g, np.abs(b - g) * 4],
     ["original", f"box, k={K}", f"gaussian, same {K}x{K} footprint",
      "|difference| x4"], ncols=4, size=3.0)
print(f"the two blurs differ by up to {np.abs(b-g).max():.3f} in intensity "
      f"(mean {np.abs(b-g).mean():.4f})")

## 1.3 The 2D kernel is an outer product 🎁🧀🧀

Here is a fact about the **2D** kernel that Part 2 depends on.

$$G(x,y)=\frac{1}{2\pi\sigma^2}e^{-\frac{x^2+y^2}{2\sigma^2}}
        =\underbrace{\tfrac{1}{\sqrt{2\pi}\sigma}e^{-\frac{x^2}{2\sigma^2}}}_{G(x)}
         \cdot\underbrace{\tfrac{1}{\sqrt{2\pi}\sigma}e^{-\frac{y^2}{2\sigma^2}}}_{G(y)}$$

The exponent splits because $x^2+y^2$ is a sum, so the 2D kernel is literally the **outer
product** of a 1D kernel with itself. Not an approximation - the same numbers.

The cell below builds a 2D Gaussian both ways and checks they are identical to machine
precision. A kernel with this property is called **separable**, and box blur is separable too
(`ones(k,k)/k**2 = outer(ones(k)/k, ones(k)/k)`).

In [ ]:
def gaussian_1d(sigma, truncate=3.0):
    # YOUR CODE HERE - the same bell in one dimension, normalised to sum 1
    raise NotImplementedError("the same bell in one dimension, normalised to sum 1")

s = 2.0
g1 = gaussian_1d(s)
K2 = gaussian_kernel(s)
outer = np.outer(g1, g1)

fig = plt.figure(figsize=(12.5, 3.6))
ax1 = fig.add_subplot(1, 4, 1)
ax1.stem(np.arange(len(g1)) - len(g1) // 2, g1)
ax1.set_title(f"1D Gaussian, sigma={s:.0f}  ({len(g1)} taps)", fontsize=10)
grid_plot(fig.add_subplot(1, 4, 2), outer, "outer(g, g)", cmap="viridis",
          vmin=0, vmax=K2.max(), fmt="", fontsize=6)
grid_plot(fig.add_subplot(1, 4, 3), K2, "the 2D kernel", cmap="viridis",
          vmin=0, vmax=K2.max(), fmt="", fontsize=6)
# NOTE the shared colour scale: autoscaling this last panel would turn 1e-16 rounding
# noise into a dramatic pattern and make identical arrays look different.
grid_plot(fig.add_subplot(1, 4, 4), np.abs(outer - K2),
          f"|difference|, same scale\nmax = {np.abs(outer - K2).max():.0e}",
          cmap="viridis", vmin=0, vmax=K2.max(), fmt="", fontsize=6)
fig.tight_layout(); plt.show()

print(f"max |outer(g,g) - 2D kernel| = {np.abs(outer - K2).max():.2e}")
assert np.allclose(outer, K2, atol=1e-15), "the 2D Gaussian must be an outer product"
print("the 2D Gaussian IS the outer product of two 1D Gaussians  OK")
print(f"\ntaps: 2D kernel = {K2.size}, two 1D passes = {2*len(g1)}"
      f"  ->  {K2.size/(2*len(g1)):.1f}x fewer")

In [ ]:
def convolve2d_rect(img, kernel, pad_mode="edge"):
    """Tap loop allowing NON-square kernels - a (1 x k) row or a (k x 1) column.

    convolve2d_fast pads kh//2 on both axes, which is only right for a square kernel.
    """
    img = np.asarray(img, float); kernel = np.asarray(kernel, float)
    kh, kw = kernel.shape
    p = np.pad(img, ((kh // 2, kh // 2), (kw // 2, kw // 2)), mode="edge")
    out = np.zeros_like(img, dtype=float)
    H, W = img.shape
    for i in range(kh):
        for j in range(kw):
            if kernel[i, j]:
                out += kernel[i, j] * p[i:i + H, j:j + W]
    return out


def gaussian_blur_separable(img, sigma, truncate=3.0):
    """Blur with a row pass, then a column pass. Works on 2D or HxWx3."""
    # YOUR CODE HERE - a row pass then a column pass, via convolve2d_rect
    raise NotImplementedError("a row pass then a column pass, via convolve2d_rect")

# the two passes, shown separately, so "separable" is something you can see
row_only = convolve2d_rect(pom_g, gaussian_1d(6)[None, :])
col_only = convolve2d_rect(pom_g, gaussian_1d(6)[:, None])
both = gaussian_blur_separable(pom_g, 6)
full2d = convolve2d_fast(pom_g, gaussian_kernel(6))
show([pom_g, row_only, both, full2d],
     ["original", "pass 1: horizontal only", "pass 2: then vertical",
      "the full 2D kernel, for comparison"], ncols=4, size=3.0)
print(f"max |separable - full 2D| = {np.abs(both - full2d).max():.2e}  -> identical")

## 1.4 Motion Blur - a kernel with a direction 🧀

Blur along a line instead of a disc, and you get the streak of a moving camera. This kernel is
**not** rotationally symmetric, and that is the whole point.

Two knobs: the **length** of the streak and its **angle**.

In [ ]:
def motion_blur_kernel(length=15, angle_deg=45.0):
    k = np.zeros((length, length))
    c = length // 2
    t = np.deg2rad(angle_deg)
    for d in np.linspace(-c, c, length * 4):
        y = int(round(c - d * np.sin(t)))
        x = int(round(c + d * np.cos(t)))
        if 0 <= y < length and 0 <= x < length:
            k[y, x] = 1.0
    return k / k.sum()


angles = (0, 45, 90, 135)
fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.3))
for ax, a in zip(axes, angles):
    grid_plot(ax, motion_blur_kernel(15, a), f"angle = {a} deg", cmap="viridis",
              fmt="", fontsize=6)
fig.suptitle("Motion-blur kernels - the line IS the kernel", fontsize=12, y=1.03)
fig.tight_layout(); plt.show()

show([pom_g] + [convolve2d_fast(pom_g, motion_blur_kernel(21, a)) for a in angles],
     ["original"] + [f"{a} deg, length 21" for a in angles], ncols=5, size=2.7,
     suptitle="Same length, four directions")
show([pom_g] + [convolve2d_fast(pom_g, motion_blur_kernel(L, 45)) for L in (7, 15, 31)],
     ["original"] + [f"length {L}, 45 deg" for L in (7, 15, 31)], ncols=4, size=3.0,
     suptitle="Same direction, three lengths")

## 1.5 Sharpen - the filter that is secretly a blur 🧀

Photoshop's sharpen is an **unsharp mask**, and the name gives it away - it sharpens by
subtracting a *blurred* copy:

$$\text{sharp} = \text{orig} + \text{amount}\cdot(\text{orig} - \text{blur}(\text{orig}))$$

**Why does subtracting a blur sharpen?** `orig - blur` is exactly the detail the blur threw
away - a **high-pass** filter. Adding it back amplifies precisely the frequencies the blur
attenuated. That is L16's Fourier aside, in two lines of code.

Two knobs: the blur's `sigma` (which *scale* of detail you boost) and `amount` (how hard).

In [ ]:
detail = pom_g - convolve2d_fast(pom_g, gaussian_kernel(2))
show([pom_g, convolve2d_fast(pom_g, gaussian_kernel(2)), detail + 0.5],
     ["original", "blurred (sigma=2)", "orig - blur = the detail  (+0.5 to see it)"],
     ncols=3, size=3.4)

amounts = (0.5, 1.5, 3.0, 6.0)
show([pom_g] + [np.clip(pom_g + a * detail, 0, 1) for a in amounts],
     ["original"] + [f"amount = {a}" for a in amounts], ncols=5, size=2.7,
     suptitle="Turning the sharpen knob - watch the halos appear")

print("over-sharpening is measurable, not just visible:")
for a in amounts:
    raw = pom_g + a * detail
    print(f"  amount={a:>4}:  {100*np.mean((raw < 0) | (raw > 1)):5.2f}% of pixels "
          f"pushed outside [0,1] and clipped")

# which scale of detail you boost
show([pom_g] + [np.clip(pom_g + 2.0 * (pom_g - convolve2d_fast(pom_g, gaussian_kernel(s))), 0, 1)
                for s in (1, 3, 8)],
     ["original"] + [f"amount=2, sigma={s}" for s in (1, 3, 8)], ncols=4, size=3.0,
     suptitle="Same amount, different detail scale")

## 1.6 Emboss - a directional difference 🧀

Emboss subtracts one side of each pixel from the other along a diagonal, producing a
lit-from-one-side relief. Rotate the kernel and the light moves.

Note this kernel sums to **1**, not 0, so flat regions keep their brightness instead of going
black. That is a deliberate choice - compare with Sobel below.

In [ ]:
EMBOSS = np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], dtype=float)
rots = [("top-left light", EMBOSS),
        ("top-right", np.rot90(EMBOSS, 1)),
        ("bottom-right", np.rot90(EMBOSS, 2)),
        ("bottom-left", np.rot90(EMBOSS, 3))]

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (name, k) in zip(axes, rots):
    grid_plot(ax, k, f"{name}  (sum={k.sum():.0f})", cmap="coolwarm",
              vmin=-2, vmax=2, textcolor="black")
fig.tight_layout(); plt.show()

show([pom_g] + [convolve2d_fast(pom_g, k) for _, k in rots],
     ["original"] + [n for n, _ in rots], ncols=5, size=2.7,
     suptitle="Emboss: rotating the kernel moves the light source")

## 1.7 Find Edges - and why one kernel is not enough 🧀🧀

`Sobel X` has a negative column and a positive column, so it fires on **vertical** edges.
`Sobel Y` is its transpose and fires on **horizontal** ones. Neither is an edge detector on
its own.

Photoshop's *Find Edges* is the **magnitude** $\sqrt{G_x^2+G_y^2}$ - and because both are
available you also get the edge **direction** $\arctan(G_y/G_x)$ for free.

Both kernels sum to **0**, so they are blind to flat brightness and report only change. An
8-bit image cannot store the negative values they produce, which is exactly what Photoshop's
**offset** field (+128) is for.

In [ ]:
SOBEL_Y = SOBEL_X.T
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2))
grid_plot(axes[0], SOBEL_X, f"Sobel X  (sum={SOBEL_X.sum():.0f})", cmap="coolwarm",
          vmin=-2, vmax=2, textcolor="black")
grid_plot(axes[1], SOBEL_Y, f"Sobel Y  (sum={SOBEL_Y.sum():.0f})", cmap="coolwarm",
          vmin=-2, vmax=2, textcolor="black")
fig.tight_layout(); plt.show()

gx = convolve2d_fast(pom_g, SOBEL_X)
gy = convolve2d_fast(pom_g, SOBEL_Y)
mag = np.sqrt(gx ** 2 + gy ** 2)
show([pom_g, np.abs(gx), np.abs(gy), mag / mag.max()],
     ["original", "|Sobel X|  vertical edges", "|Sobel Y|  horizontal edges",
      "magnitude = Find Edges"], ncols=4, size=3.0)

strong = mag > np.percentile(mag, 99)
ang = np.rad2deg(np.arctan2(np.abs(gy), np.abs(gx)))[strong]
print("orientation of the strongest 1% of edges:")
for lo, hi, lab in [(0, 30, "near-vertical   (X sees it)"),
                    (30, 60, "diagonal        (needs both)"),
                    (60, 90, "near-horizontal (Y sees it)")]:
    print(f"  {lab:<30}{100*np.mean((ang >= lo) & (ang < hi)):5.1f}%")
print(f"\nedge energy recovered by one axis alone: "
      f"X {100*np.abs(gx)[strong].sum()/mag[strong].sum():.0f}%, "
      f"Y {100*np.abs(gy)[strong].sum()/mag[strong].sum():.0f}%")
print("On a perfect diagonal each axis sees only 1/sqrt(2) = 71% of the gradient.")

## 1.8 Filter > Other > Custom 🧀

Now the actual Photoshop dialog: type your own numbers, pick a scale and an offset.

Two rules you can now derive rather than memorise:

- weights summing to **1** preserve average brightness (any blur),
- weights summing to **0** ignore flat regions and respond only to change (any edge detector).

Try your own. The cell below shows three: an edge-enhancing kernel, an outline detector, and
one designed to do nothing except through the offset.

In [ ]:
# YOUR CODE HERE - design three of your own 3x3 kernels.
# Make at least one sum to 1 and at least one sum to 0, and predict what each will do
# BEFORE you run the next cell.
CUSTOM = {
    "my kernel 1 (sum=?)": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float),
    "my kernel 2 (sum=?)": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float),
    "my kernel 3 (sum=?)": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float),
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, (n, k) in zip(axes, CUSTOM.items()):
    grid_plot(ax, k, f"{n}", cmap="coolwarm", vmin=-8, vmax=8, textcolor="black")
fig.tight_layout(); plt.show()

outs, titles = [pom_g], ["original"]
for n, k in CUSTOM.items():
    r = convolve2d_fast(pom_g, k)
    if abs(k.sum()) < 1e-9:
        r = r + 0.5                     # the Photoshop offset field
    outs.append(r); titles.append(n + ("  +offset" if abs(k.sum()) < 1e-9 else ""))
show(outs, titles, ncols=4, size=3.0)

---
# Part 2 - the thumbnail generator 🧀🧀

Open any lo-fi music upload on YouTube and you get the same layout, for example
[this one](https://www.youtube.com/watch?v=ydIFRGlXGy4):

- **background** - the cover art, blown up to fill 1280x720, blurred hard, darkened
- **foreground** - the untouched sharp cover, about 25% of the width, centred
- **text** - bold white title, lighter grey artist below

We are going to build it **one layer at a time** and turn each knob, rather than producing the
finished thing in one cell. Each step below is a complete, viewable image.

In [ ]:
def center_square(img):
    h, w = img.shape[:2]
    s = min(h, w)
    return img[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s]


def resize(img, size_hw):
    """size_hw = (H, W). PIL for resampling only - the convolution stays ours."""
    pil = Image.fromarray((np.clip(img, 0, 1) * 255).astype(np.uint8))
    return np.asarray(pil.resize((size_hw[1], size_hw[0]), Image.LANCZOS), float) / 255.0


def render(canvas_rgb, title=None, artist=None, note="", figsize=(8, 4.5)):
    """Draw a 1280x720 array, optionally with the two text lines on top."""
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.imshow(np.clip(canvas_rgb, 0, 1)); ax.axis("off"); ax.set_position([0, 0, 1, 1])
    if title:
        ax.text(0.5, 0.255, title, transform=ax.transAxes, ha="center", va="top",
                color="white", fontsize=12, fontweight="bold")
    if artist:
        ax.text(0.5, 0.185, artist, transform=ax.transAxes, ha="center", va="top",
                color="0.78", fontsize=10.5)
    if note:
        ax.text(0.012, 0.02, note, transform=ax.transAxes, ha="left", va="bottom",
                color="#F2A800", fontsize=8)
    plt.show()


H, W = CANVAS
cover = center_square(pom_full)
SIDE = int(0.25 * W)
TOP = int(H * 0.42) - SIDE // 2
LEFT = (W - SIDE) // 2
fg = resize(cover, (SIDE, SIDE))
TITLE, ARTIST = "Ten more times", "Alis"
print(f"canvas {W}x{H}; cover art inset {SIDE}x{SIDE} px at ({TOP},{LEFT})")

### Step 1 - just the cover, on a flat canvas

Nothing but the sharp square, centred and slightly above the middle. Already readable, and
already boring: the 16:9 canvas is mostly empty.

In [ ]:
step1 = np.full((H, W, 3), 0.12)
step1[TOP:TOP + SIDE, LEFT:LEFT + SIDE] = fg
render(step1, note="step 1: cover art on a flat background")

### Step 2 - add the text

Title in bold white, artist below it in grey. The grey is doing real work: it creates a
hierarchy, so your eye reads the title first.

In [ ]:
render(step1, TITLE, ARTIST, note="step 2: + title and artist")

### Step 3 - fill the background with the cover itself

Scale the square up until it **covers** the whole 16:9 canvas and crop the overflow.

And now the problem the blur exists to solve: the background competes with the foreground.
Two copies of the same sharp image at two scales is visually noisy, and the eye does not know
where to land.

In [ ]:
scale = max(W / cover.shape[1], H / cover.shape[0])
big = resize(cover, (int(round(cover.shape[0] * scale)), int(round(cover.shape[1] * scale))))
y0, x0 = (big.shape[0] - H) // 2, (big.shape[1] - W) // 2
bg_sharp = big[y0:y0 + H, x0:x0 + W]

step3 = bg_sharp.copy()
step3[TOP:TOP + SIDE, LEFT:LEFT + SIDE] = fg
render(step3, TITLE, ARTIST, note="step 3: + background, unblurred - too busy")

### Step 4 - turn the blur knob

Now blur the background. This is the knob that matters, so try several values rather than
jumping to an answer.

Watch for the point where the background stops reading as *an image* and starts reading as
*texture*. Below it, the background still competes. Far above it, you lose the colour that
made the thumbnail feel like the cover.

In [ ]:
for sig in (5, 15, 30, 60):
    t0 = time.perf_counter()
    bgb = gaussian_blur_separable(bg_sharp, sig)
    dt = time.perf_counter() - t0
    frame = bgb.copy()
    frame[TOP:TOP + SIDE, LEFT:LEFT + SIDE] = fg
    k = int(6 * sig) + 1
    render(frame, TITLE, ARTIST, figsize=(6.6, 3.7),
           note=f"sigma = {sig}   (kernel {k}x{k} = {k*k:,} taps/px)   blurred in {dt:.1f}s")

### Step 5 - turn the darkening knob

The blur alone is not enough: the background is still as bright as the foreground, so the
cover art does not pop and the white text has nothing to sit against.

Multiply the background down. Part 3 explains *why multiplying is the right operation* and
what the alternative would have done.

In [ ]:
bg30 = gaussian_blur_separable(bg_sharp, 30)
for f in (1.0, 0.75, 0.55, 0.35):
    frame = bg30 * f
    frame[TOP:TOP + SIDE, LEFT:LEFT + SIDE] = fg
    render(frame, TITLE, ARTIST, figsize=(6.6, 3.7),
           note=f"background x {f}")

### Step 6 - the finished thumbnail

`sigma = 30`, background at `0.55`, cover at 25% of the width. Compare it with the reference
at the top of this Part.

In [ ]:
thumb = bg30 * 0.55
thumb[TOP:TOP + SIDE, LEFT:LEFT + SIDE] = fg
render(thumb, TITLE, ARTIST, figsize=(9, 5.06))

### Task 2.1 - what did that blur actually cost? 🧀🧀

`sigma=30` means a `181x181` kernel: **32,761 taps per pixel**, on a 1280x720x3 canvas. That
is about 9x10^10 multiply-adds.

You blurred it in a couple of seconds because `gaussian_blur_separable` used the outer-product
fact from Task 1.3 - two 1D passes, `2k = 362` taps instead of `k**2 = 32,761`.

The cell below runs all three implementations on one 480x270 channel and scales up. **Predict
first: at this kernel size, does the tap loop still beat the pixel loop by 100x, the way it did
at k=3 in Part 0?**

In [ ]:
SIGMA_BG, K_BG = 30.0, 181
TAPS, PIXELS = K_BG ** 2, H * W * 3
probe = rng.random((270, 480))
K2D = gaussian_kernel(SIGMA_BG)
assert K2D.shape[0] == K_BG

times = {}
for label, fn in [("naive  (pixel loop)", lambda im: convolve2d_naive(im, K2D)),
                  ("fast   (tap loop)", lambda im: convolve2d_fast(im, K2D)),
                  ("separable (two 1D)", lambda im: gaussian_blur_separable(im, SIGMA_BG))]:
    t0 = time.perf_counter(); fn(probe); times[label] = time.perf_counter() - t0

sf = PIXELS / probe.size
print(f"measured on one 480x270 channel, scaled by {sf:.0f}x to the full 3-channel canvas\n")
print(f"{'implementation':<22}{'taps/px':>9}{'measured':>11}{'full canvas':>14}")
print("-" * 56)
for label, t in times.items():
    taps = 2 * K_BG if "separable" in label else TAPS
    full = t * sf
    print(f"{label:<22}{taps:>9,}{t:>10.1f}s"
          f"{(f'{full:.1f} s' if full < 90 else f'{full/60:.1f} min'):>14}")
print("-" * 56)
tn, tf, ts = (times["naive  (pixel loop)"], times["fast   (tap loop)"],
              times["separable (two 1D)"])
print(f"tap loop vs pixel loop at k={K_BG}: {tn/tf:5.2f}x")
print(f"separable vs the better of the two: {min(tn, tf)/ts:5.1f}x")
print(f"predicted from tap counts alone   : {TAPS/(2*K_BG):5.0f}x")

**Read that table.** At `k=3` in Part 0 the tap loop beat the pixel loop by roughly two orders
of magnitude. At `k=181` they land in the same ballpark - close enough that which one wins
depends on what else your machine is doing. The advantage has evaporated.

Nothing about the arithmetic changed. What changed is where the time goes:

- the **pixel loop** reads a small `181x181` window per output pixel, and neighbouring pixels
  reuse nearly the same window, so it stays in cache,
- the **tap loop** sweeps the *entire padded image* once per tap - 32,761 full passes over
  several megabytes. It is limited by memory bandwidth, not by the multiplier.

So "just vectorise it" is a constant-factor trick that buys a lot at small `k` and quietly
stops paying at large `k`. **Separability is the only change here that survives the problem
getting bigger**, because it is the only one that does less work.

Counting operations tells you how an algorithm **scales**; only a measurement tells you what
it **costs**; and a measurement only transfers to conditions like the ones you measured under.

This factorisation returns in **L19** as *depthwise-separable convolutions* - the trick that
lets MobileNet run on a phone.

---
# Part 3 - HSV 🧀🧀

Step 5 above was `background * 0.55`. Why *multiply*, and not subtract?

This is the frame **"Three numbers, but which three?"** from L16 (page 8). RGB stores how much
of each primary. Nothing in it means *how bright* or *which colour* - both are spread across
all three numbers. **HSV** keeps the same pixel and changes the axes:

- **H**ue - the angle, 0-360 degrees: which colour
- **S**aturation - the radius: how far from grey
- **V**alue - the height: how bright

$$V=\max(R,G,B),\qquad S=\frac{\max-\min}{\max}\ \ (0 \text{ if } \max = 0)$$

and $H$ is decided by *which* channel is the max, and by how far apart the other two are.

In [ ]:
hsv = rgb_to_hsv(pom)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(pom); axes[0].set_title("RGB image", fontsize=10)
for ax, ch, name, cm in [(axes[1], hsv[..., 0], "H  hue (which colour)", "hsv"),
                         (axes[2], hsv[..., 1], "S  saturation (how pure)", "gray"),
                         (axes[3], hsv[..., 2], "V  value (how bright)", "gray")]:
    ax.imshow(ch, cmap=cm, vmin=0, vmax=1); ax.set_title(name, fontsize=10)
for ax in axes: ax.axis("off")
fig.tight_layout(); plt.show()
print("Note the H panel goes to noise wherever the image is near-grey or white.")
print("Hue is an ANGLE, and the angle is meaningless when the radius (S) is ~0.")

### Task 3.1 - implement it 🧀🧀

Write `V` and `S` yourself, then check the whole thing against matplotlib's version rather
than importing it and hoping.

In [ ]:
def rgb_to_hsv_mine(rgb):
    """RGB in [0,1] -> HSV with H in [0,1). Written out, then checked."""
    # YOUR CODE HERE - V = max(R,G,B); S = (max-min)/max; H from which channel is the max
    raise NotImplementedError("V = max(R,G,B); S = (max-min)/max; H from which channel is the max")

mine, ref = rgb_to_hsv_mine(pom), rgb_to_hsv(pom)
print("max |mine - matplotlib|: " + "  ".join(
    f"{n} {np.abs(mine[...,i]-ref[...,i]).max():.1e}" for i, n in enumerate("HSV")))
assert np.allclose(mine, ref, atol=1e-10), "HSV conversion disagrees with matplotlib"
print("HSV conversion OK")

### Task 3.2 - so which darkening was right? 🧀🧀

Two ways to darken in RGB, and they are **not** the same operation:

- **subtract** a constant - the ratios between R, G and B change, so the hue drifts, and
  everything below the constant clips to black,
- **multiply** by a factor - the ratios are preserved exactly.

Now the part worth being precise about: scaling `V` in HSV **is** the uniform RGB multiply.
Same operation, two descriptions. HSV is not "the only way to darken correctly".

What HSV buys you is that the question cannot be asked wrongly. In RGB, "make it darker" is
ambiguous and one of the two obvious readings is wrong. In HSV, brightness *is* an axis, so
moving `V` cannot do anything else.

In [ ]:
sub = np.clip(pom - 110 / 255, 0, 1)
mul = np.clip(pom * 0.55, 0, 1)
hd = hsv.copy(); hd[..., 2] *= 0.55
v_scaled = hsv_to_rgb(hd)

gap = np.abs(v_scaled - mul).max()
print(f"max |HSV V-scale - RGB multiply| = {gap:.2e}   -> literally the same operation")
assert gap < 1e-6


def hue_shift_deg(a, b, sat_floor=0.2):
    ha, hb = rgb_to_hsv(a)[..., 0], rgb_to_hsv(b)[..., 0]
    m = rgb_to_hsv(a)[..., 1] > sat_floor
    d = np.abs(hb - ha)[m] * 360
    return np.minimum(d, 360 - d)


print(f"subtract 110 : mean hue shift {hue_shift_deg(pom, sub).mean():5.1f} deg, "
      f"{100*np.mean(sub.max(axis=2)==0):4.1f}% crushed to pure black")
print(f"multiply 0.55: mean hue shift {hue_shift_deg(pom, mul).mean():5.1e} deg, "
      f"colour preserved exactly")
show([pom, sub, mul], ["original", "RGB: subtract 110",
                       "RGB: multiply 0.55  ==  HSV: V x 0.55"], ncols=3, size=3.6)

### Task 3.3 - the Hue/Saturation panel 🧀🧀

| what you do to HSV | the Photoshop menu item |
|---|---|
| `H += 30/360` | Image > Adjustments > Hue/Saturation, **Hue** slider |
| `S *= f` | same panel, **Saturation** slider |
| keep one hue band, grey the rest | the "red dress in a black-and-white film" effect |

Each is one line, because each moves exactly one axis.

In [ ]:
h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]

outs, titles = [pom], ["original"]
for deg in (60, 120, 180, 240):
    r = hsv.copy(); r[..., 0] = (h + deg / 360) % 1.0
    outs.append(hsv_to_rgb(r)); titles.append(f"H + {deg} deg")
show(outs, titles, ncols=5, size=2.7, suptitle="The Hue slider")

outs, titles = [], []
for f in (0.0, 0.5, 1.0, 1.8):
    r = hsv.copy(); r[..., 1] = np.clip(s * f, 0, 1)
    outs.append(hsv_to_rgb(r)); titles.append(f"S x {f}")
show(outs, titles, ncols=4, size=3.0, suptitle="The Saturation slider")

band = ((h < 25 / 360) | (h > 335 / 360)) & (s > 0.25)
sel = hsv.copy(); sel[..., 1] = np.where(band, s, 0.0)
show([pom, band, hsv_to_rgb(sel)],
     ["original", f"hue band mask ({100*band.mean():.0f}% of pixels)",
      "keep the reds, grey the rest"], ncols=3, size=3.6)

---
## Where this goes next

You now have a convolution engine, a filter menu, and a colour space in which "which colour"
and "how bright" are separate numbers.

**HW1c** points all three at a real **liver biopsy** and asks them to measure something a
pathologist cares about. It works - and then you move to a second slide and it stops working,
which is the argument for everything after L16.

**Also next:** HW1 Part B trains a small CNN on Fashion-MNIST and shows you its first-layer
kernels, so you can put your hand-designed zoo next to a set the network chose for itself.

In [ ]:
print("Kernels you built in this notebook:")
for i, n in enumerate(["box blur (k = 3..41)", "Gaussian blur (sigma = 1..8)",
                       "motion blur (4 angles, 3 lengths)", "unsharp mask (4 amounts, 3 scales)",
                       "emboss (4 rotations)", "Sobel X / Y / magnitude",
                       "3 custom kernels"], 1):
    print(f"  {i}. {n}")
print("\nEvery weight in every one of them was chosen by a human. HW1c is where that becomes"
      "\na problem.")